# 04 — BPE on a real corpus (Part D)

The toy corpus verifies correctness; here you measure what BPE *does* at scale on **Tiny Shakespeare** (~1.1 M characters). Training 1000 merges takes about 1–3 minutes: develop with `DEV = True`, then run once.

**How the 🧩 exercises work.** Cells marked 🧩 contain `____` blanks: complete them using the hints (they name the Python functions you need). Each exercise is followed by a ✅ *check* cell that tells you immediately whether you got it right. When an exercise matches a function of `src/`, the docstring of that function says so: you then **move your working code into `src/`** and run `pytest`.

In [ ]:
import sys, pathlib, json, time
ROOT = pathlib.Path.cwd().parent          # notebooks/ is one level below the repo root
sys.path.insert(0, str(ROOT))
import pandas as pd
import matplotlib.pyplot as plt
%load_ext autoreload
%autoreload 2

from collections import Counter
from src.bpe import BPETokenizer
from src.preprocessing import preprocess_text
from src.evaluation import (split_lines, avg_tokens_per_word, compression_ratio, unseen_words, oov_rate_words, token_length_histogram)

In [ ]:
raw = (ROOT / "data/real/tinyshakespeare.txt").read_text(encoding="utf-8").splitlines()
lines = [l for l in (preprocess_text(x) for x in raw) if l]
train_lines, test_lines = split_lines(lines, test_fraction=0.2, seed=0)
DEV = False                      # set to True while developing
if DEV: train_lines, test_lines = train_lines[:3000], test_lines[:800]
print(len(train_lines), "train lines |", len(test_lines), "test lines")
print("whole-word OOV rate of test vs train:", round(oov_rate_words(train_lines, test_lines), 4))

## Train with several merge budgets

In [ ]:
BUDGETS = [100, 500, 1000]
models, rows = {}, []
for n in BUDGETS:
    t0 = time.time(); t = BPETokenizer().train(train_lines, num_merges=n); train_time = time.time() - t0
    models[n] = t
    rows.append({"merges": n, "vocab_size": t.vocab_size, "tokens/word (test)": avg_tokens_per_word(t, test_lines),
                 "chars/token (test)": compression_ratio(t, test_lines), "train time (s)": train_time})
results = pd.DataFrame(rows); results.to_csv(ROOT / "results/bpe_real_results.csv", index=False); results

## Measure it yourself

`src/evaluation.py` gives you ready-made metrics. To understand them, recompute two of them with plain Python.

### 🧩 Exercise 4.1 — Fertility and characters per token by hand

On 500 test lines, compute the **average number of tokens per word** (fertility) and the **characters per token**, for the BPE with the most merges, and compare with the provided functions.

Replace every `____` with the right code, run the cell, then run the **check** cell below it.

**Hints**
* Words of a line: `line.split()`. Tokens of a line: `tokenizer.encode(line)` (a list).
* Fertility = total tokens / total words. Characters per token = total characters of the words / total tokens.
* Use **totals** over all lines (not an average of per-line ratios).

In [ ]:
t = models[max(BUDGETS)]
sample = test_lines[:500]

n_words  = sum(len(line.____()) for line in sample)
n_tokens = sum(len(t.____(line)) for line in sample)
n_chars  = sum(len(w) for line in sample for w in line.split())

fertility = ____
chars_per_token = ____
print(round(fertility, 3), round(chars_per_token, 3))

In [ ]:
assert abs(fertility - avg_tokens_per_word(t, sample)) < 1e-12
assert abs(chars_per_token - compression_ratio(t, sample)) < 1e-12
print("✅ Exercise 4.1 OK")

### 🧩 Exercise 4.2 — Words never seen in training

Compute the fraction of test word **occurrences** whose word never appears in the training lines, and list a few of the distinct unseen words.

Replace every `____` with the right code, run the cell, then run the **check** cell below it.

**Hints**
* Build the training vocabulary as a **set** of words: `{w for line in train_lines for w in line.split()}`.
* `w not in train_vocab` is O(1) with a set (it would be painfully slow with a list).
* Occurrence rate = (number of unseen test word occurrences) / (number of test word occurrences).

In [ ]:
train_vocab = {w for line in train_lines for w in line.split()}
test_words = [w for line in test_lines for w in line.split()]

unseen_occurrences = sum(1 for w in test_words if w ____ train_vocab)
oov_occ = ____
distinct_unseen = sorted({w for w in test_words if w not in train_vocab})
print(round(oov_occ, 4), len(distinct_unseen), distinct_unseen[:8])

In [ ]:
assert abs(oov_occ - oov_rate_words(train_lines, test_lines)) < 1e-12
assert distinct_unseen == unseen_words(train_lines, test_lines)
print("✅ Exercise 4.2 OK")

### 🧩 Exercise 4.3 — How long are the learned tokens?

Count how many vocabulary tokens have each length (in characters), then plot the histogram.

Replace every `____` with the right code, run the cell, then run the **check** cell below it.

**Hints**
* `Counter(len(tok) for tok in vocabulary)`.
* The vocabulary of your tokenizer is `t.token_to_id` (a dict whose keys are tokens); skip `"[UNK]"`.
* Replace the end-of-word marker `</w>` by one character before measuring, so that it counts as 1: `tok.replace("</w>", "_")`.
* `plt.bar(x_values, heights)`.

In [ ]:
lengths = ____(len(tok.____("</w>", "_")) for tok in t.token_to_id if tok != "[UNK]")
xs = sorted(lengths)
plt.bar(xs, [lengths[x] for x in xs])
plt.xlabel("token length (characters)"); plt.ylabel("number of tokens in vocab"); plt.title(f"Vocabulary token lengths (BPE-{max(BUDGETS)})")
plt.savefig(ROOT / "results/figures/token_lengths.png", dpi=150); plt.show()

In [ ]:
assert sum(lengths.values()) == t.vocab_size - 1
assert lengths == token_length_histogram(t)
print("✅ Exercise 4.3 OK")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
ax[0].plot(results.merges, results["tokens/word (test)"], marker="o"); ax[0].set_xlabel("number of merges"); ax[0].set_ylabel("tokens per word (test)")
ax[1].plot(results.merges, results["chars/token (test)"], marker="o", color="tab:green"); ax[1].set_xlabel("number of merges"); ax[1].set_ylabel("characters per token (test)")
plt.tight_layout(); plt.savefig(ROOT / "results/figures/bpe_real_curves.png", dpi=150); plt.show()

## What did BPE learn?

In [ ]:
print("First 25 merges: ", [f"{a}|{b}" for a, b in t.merges[:25]])
print("Merges 500-515:  ", [f"{a}|{b}" for a, b in t.merges[500:515]])
print("Frequency of the chosen pair at merge 1, 10, 100, 500, 1000:", [t.merge_frequencies[i - 1] for i in [1, 10, 100, 500, 1000] if i <= len(t.merge_frequencies)])
print("Longest tokens:", sorted(t.token_to_id, key=len, reverse=True)[:15])

## Unseen words

In [ ]:
import random
rare = unseen_words(train_lines, test_lines); random.Random(1).shuffle(rare)
for w in rare[:15]: print(f"{w:18}", {n: models[n].encode_word(w) for n in (BUDGETS[0], BUDGETS[-1])})

## Questions (include the numbers in your report)

1. How do vocabulary size, tokens/word and characters/token change with the number of merges? Where do you see diminishing returns?
2. Which are the first merges? Do they look linguistically meaningful? What about merges 500–515?
3. How are *unseen* words segmented? Compare BPE-100 and BPE-1000, and the whole-word OOV rate printed above.
4. Compare BPE-1000 with the character and word tokenizers of notebook 02 (vocab size and sequence length).
5. Does more merges always help? (think of the embedding matrix, Part F, and rare tokens)

**Your answer:**

_(write here)_

## Optional extensions (bonus, clearly marked in your report)

* **Punctuation:** `speak,` and `speak.` are different "words". Separate punctuation with `re.sub(r"([.,!?;:])", r" \1 ", line)` before training and measure the effect.
* **Speed:** your `train` recomputes all pair counts at every merge. Implement an incremental update and report the speed-up.
* **Case:** train on lower-cased text and compare the vocabulary.

In [ ]:
# your optional experiments